# Final ML Project: Stroke Prediction Classification
Student: Mahdi Mohammadkhani

This notebook follows the classification requirements for the final ML project.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt

## 2. Load Dataset

In [ ]:
df = pd.read_csv("../data/healthcare-dataset-stroke-cleaned.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
print(df["stroke"].value_counts())
print(df.isna().sum())

## 3. Split Data into Train, Validation, and Test Sets
The split is 70% train, 15% validation, and 15% test. Stratification preserves the class balance.

In [ ]:
X = df.drop(columns=["stroke"])
y = df["stroke"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

## 4. Preprocessing Pipeline
The dataset is already cleaned. The remaining categorical column is one-hot encoded and numeric features are scaled inside the model pipeline.

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

def make_preprocessor():
    return ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numeric", StandardScaler(), numeric_cols)
    ])

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

## 5. Train Required Classification Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(max_depth=6, min_samples_leaf=15, class_weight="balanced", random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=10, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting Classifier": GradientBoostingClassifier(n_estimators=180, learning_rate=0.05, max_depth=2, random_state=42),
    "K-Nearest Neighbors Classifier": KNeighborsClassifier(n_neighbors=15, weights="distance"),
    "Support Vector Classifier": SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", probability=True, random_state=42)
}

def evaluate(name, split, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Split": split,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob)
    }

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
fitted_models = {}
rows = []

for name, clf in models.items():
    pipe = Pipeline([("preprocess", make_preprocessor()), ("model", clf)])
    if name == "Gradient Boosting Classifier":
        pipe.fit(X_train, y_train, model__sample_weight=sample_weights)
    else:
        pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    for split_name, X_split, y_split in [("Validation", X_val, y_val), ("Test", X_test, y_test)]:
        y_pred = pipe.predict(X_split)
        y_prob = pipe.predict_proba(X_split)[:, 1]
        rows.append(evaluate(name, split_name, y_split, y_pred, y_prob))

results = pd.DataFrame(rows)
results.round(4)

## 6. Ensemble Models
The top three models by validation ROC-AUC are used for soft voting and Bayesian weighted averaging.

In [ ]:
top3 = results[results["Split"] == "Validation"].sort_values("ROC-AUC", ascending=False).head(3)["Model"].tolist()
validation_auc = results[(results["Split"] == "Validation") & (results["Model"].isin(top3))].set_index("Model").loc[top3, "ROC-AUC"].values
bayesian_weights = validation_auc / validation_auc.sum()
print("Top 3 models:", top3)
print("Bayesian weights:", dict(zip(top3, bayesian_weights.round(4))))

ensemble_rows = []
for split_name, X_split, y_split in [("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    prob_matrix = np.column_stack([fitted_models[m].predict_proba(X_split)[:, 1] for m in top3])

    soft_prob = prob_matrix.mean(axis=1)
    soft_pred = (soft_prob >= 0.5).astype(int)
    ensemble_rows.append(evaluate("Soft Voting Ensemble (Top 3)", split_name, y_split, soft_pred, soft_prob))

    bayes_prob = prob_matrix.dot(bayesian_weights)
    bayes_pred = (bayes_prob >= 0.5).astype(int)
    ensemble_rows.append(evaluate("Bayesian Weighted Ensemble (Top 3)", split_name, y_split, bayes_pred, bayes_prob))

final_results = pd.concat([results, pd.DataFrame(ensemble_rows)], ignore_index=True)
final_results.round(4).sort_values(["Split", "F1-Score"], ascending=[True, False])

## 7. Save Final Comparison Table

In [ ]:
final_results.round(4).to_csv("../results/model_metrics.csv", index=False)
final_results.round(4)

## 8. Conclusion
For this imbalanced healthcare dataset, F1-Score, Recall, and ROC-AUC are more informative than accuracy alone. The ensemble model achieved the strongest test F1-Score because it combined the strengths of Logistic Regression, Gradient Boosting, and Random Forest probability predictions.